# P123 — SentencePiece: un tokenizador y detokenizador de subpalabras simple e independiente del idioma

## 1. Título y paper

**Paper:** *SentencePiece: A simple and language independent subword tokenizer and detokenizer for Neural Text Processing*  
**Autoría:** Taku Kudo, John Richardson  
**Año y venue:** 2018 · EMNLP 2018 (demos), 66–71  
**Nivel:** L2 · **Motor:** `sentencepiece`  
**Ficha completa:** [`P123_sentencepiece`](../../papers/foundational/P123_sentencepiece/README.md)

**Hito:** Elimina la pretokenización por espacios y hace la detokenización exacta, lo que convierte al tokenizador en una pieza reproducible e independiente del idioma.

- [doi:10.18653/v1/D18-2012](https://doi.org/10.18653/v1/D18-2012)

> Este notebook implementa una **miniatura** del mecanismo. No reproduce el experimento original ni sus métricas: reproduce la idea para que se pueda inspeccionar y discutir.


## 2. Objetivos

1. Explicar qué problema resolvió el paper: BPE suponía texto ya partido por espacios, y eso no es universal: el japonés y el chino no los usan. Además cada implementación normalizaba a su manera, así que reconstruir el texto original era imposible y los resultados no eran comparables.
2. Ejecutar una implementación mínima de la propuesta: Tratar la entrada como un flujo de caracteres crudo, codificar el espacio como un símbolo más del vocabulario, y ofrecer también un modelo unigrama donde la segmentación es inferencia probabilística y se puede muestrear para regularizar.
3. Predecir el resultado antes de ejecutar, y contrastar la predicción con la salida.
4. Identificar al menos una limitación de la miniatura y una del paper original.
5. Conectar el hito con el siguiente eslabón de la ruta.


## 3. Prerrequisitos

- Python 3.11+ y el paquete del programa instalado (`pip install -e .`).
- Haber leído la guía [método de lectura en 5 pasadas](../../papers/guides/METODO_DE_LECTURA_EN_5_PASADAS.md).
- Hitos previos:
- P118


## 4. Intuición

BPE suponía que el texto ya venía partido por espacios. El japonés no los usa, y un texto con espacio doble ya no se puede reconstruir. La solución es no suponer nada: tratar la entrada como un flujo de caracteres.


## 5. Concepto mínimo

```text
Por espacios : "el  gato" → ["el", "gato"] → "el gato"   ✗ irreversible
Flujo crudo  : "el  gato" → [e,l,▁,▁,g,a,t,o]      ✓ exacto

Modelo unigrama: la segmentación es INFERENCIA, no una regla
```


## 6. Código explicado

El motor aísla el mecanismo del paper con datos de juguete y salida inspeccionable.


In [ ]:
import json
import pathlib
import sys

ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from ai_evolution.papers_lab import run_paper_lab


def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


In [ ]:
r = run_paper_lab('sentencepiece', seed=7)['result']
show(r)

## 7. Predicción antes de ejecutar

1. ¿Cuántos de los tres textos reconstruye cada método?
2. ¿Qué pasa con el japonés al partir por espacios?
3. ¿Cuántas segmentaciones admite una palabra?

> Escribe tu respuesta aquí antes de continuar.


## 8. Experimento controlado

Se varía una sola cosa y se observa el efecto.


In [ ]:
for semilla in (1, 7, 42):
    r = run_paper_lab('sentencepiece', seed=semilla)
    print(f'semilla {semilla:>2} · evidencia principal:')
    for e in r['evidence']:
        print('   +', e)
    break  # determinista: basta una para ver la estructura
for semilla in (1, 7, 42):
    r = run_paper_lab('sentencepiece', seed=semilla)['result']
    print(f'semilla {semilla:>2} → claves: {list(r)[:4]}')

## 9. Salida interpretable

Partir por espacios reconstruye **2 de 3**; el flujo crudo, **3 de 3**. En japonés, partir por espacios da **1 pieza** para la frase entera —inútil— y el flujo crudo da **6**, sin necesitar saber en qué idioma está. Y «internacional» admite **5 segmentaciones** distintas, de log-probabilidad −2,4 la mejor a −18,9 la peor.


## 10. Comentario pedagógico

Que haya varias segmentaciones válidas no es un defecto: muestrear entre ellas durante el entrenamiento es **regularización de subpalabra**, y mejora la robustez ante erratas y variantes. Es la diferencia entre un tokenizador determinista y uno probabilístico.


## 11. Error o anti-patrón deliberado

Anti-patrón: tratar el tokenizador como preprocesado que no hay que versionar.


In [ ]:
print('El tokenizador es parte del modelo: cambiarlo invalida los pesos.')
print('Y si no es reversible, no se puede auditar que entro exactamente al modelo.')
print('Versionalo con el checkpoint, no con el codigo de datos.')

## 12. Corrección

Reversibilidad y ambigüedad, medidas:


In [ ]:
r = run_paper_lab('sentencepiece', seed=3)['result']
for fila in r['comparacion']:
    print(fila)
print('segmentaciones posibles:', r['segmentaciones_posibles'])
for s in r['mejores_segmentaciones']:
    print('  ', s)

## 13. Desafío guiado

Explica por qué la reversibilidad exacta importa para auditar un sistema en producción, y qué coste tiene en longitud de secuencia.


In [ ]:
r = run_paper_lab('sentencepiece', seed=3)['result']
show(r)

## 14. Desafío autónomo

Tokeniza el mismo texto en español, en un idioma sin espacios y con emojis. Compara piezas por carácter en los tres casos.


## 15. Evidencia de aprendizaje

Guarda la comparación y qué idioma sale peor parado con tu tokenizador.

Autoevaluación y respuestas esperadas: [ficha del paper](../../papers/foundational/P123_sentencepiece/README.md) · evaluación formal: [`assessments/papers/P123_sentencepiece.md`](../../assessments/papers/P123_sentencepiece.md)


## 16. Cierre

Con el texto ya resuelto, quedan los grafos y los documentos: dos formas de entrada donde la estructura es tan informativa como el contenido.


## 17. Conexión con el siguiente hito



Ruta completa: [`papers/ROADMAP.md`](../../papers/ROADMAP.md)
